In [1]:
# Load the churn dataset and do an initial inspection
import pandas as pd
import numpy as np


df= pd.read_csv("Customer Churn.csv", encoding='ascii')

print(df.head())
print(df.shape)
print(df.isna().sum().sort_values(ascending=False).head(10))

   customerID  gender  SeniorCitizen Partner Dependents  tenure PhoneService  \
0  7590-VHVEG  Female              0     Yes         No       1           No   
1  5575-GNVDE    Male              0      No         No      34          Yes   
2  3668-QPYBK    Male              0      No         No       2          Yes   
3  7795-CFOCW    Male              0      No         No      45           No   
4  9237-HQITU  Female              0      No         No       2          Yes   

      MultipleLines InternetService OnlineSecurity  ... DeviceProtection  \
0  No phone service             DSL             No  ...               No   
1                No             DSL            Yes  ...              Yes   
2                No             DSL            Yes  ...               No   
3  No phone service             DSL            Yes  ...              Yes   
4                No     Fiber optic             No  ...               No   

  TechSupport StreamingTV StreamingMovies        Contract Pape

In [3]:
# 1) strip whitespace in object columns

obj_cols = df.select_dtypes(include=['object']).columns
for col in obj_cols:
    df[col] = df[col].astype(str).str.strip()

In [4]:
# 2) Convert TotalCharges to numeric (in this dataset, blanks can appear and should become NaN)
# Treat common blank tokens as missing

blank_tokens = ['', ' ', 'NA', 'N/A', 'nan', 'NaN', 'None']
df['TotalCharges'] = df['TotalCharges'].replace(blank_tokens, np.nan)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')

In [ ]:
# 3) Ensure numeric columns are numeric

df['tenure'] = pd.to_numeric(df['tenure'], errors='coerce')
df['MonthlyCharges'] = pd.to_numeric(df['MonthlyCharges'], errors='coerce')

In [ ]:
# 4) Deduplicate rows (keep first)

rows_before = df.shape[0]
df = df.drop_duplicates()
rows_after = df.shape[0]

In [ ]:
# 5) Impute missing TotalCharges in a practical way:
# If tenure == 0, TotalCharges should be 0. Otherwise, fill with MonthlyCharges * tenure (a common approximation)

missing_total_before = int(df['TotalCharges'].isna().sum())
mask_missing_total = df['TotalCharges'].isna()
mask_tenure_zero = mask_missing_total & (df['tenure'].fillna(-1) == 0)
df.loc[mask_tenure_zero, 'TotalCharges'] = 0.0

mask_still_missing = df['TotalCharges'].isna()
df.loc[mask_still_missing, 'TotalCharges'] = (
    df.loc[mask_still_missing, 'MonthlyCharges'] * df.loc[mask_still_missing, 'tenure']
)
missing_total_after = int(df['TotalCharges'].isna().sum())


In [ ]:
# 6) Standardize yes/no style columns and churn target to consistent capitalization

yn_cols = ['Partner','Dependents','PhoneService','PaperlessBilling','Churn']
for col in yn_cols:
    df[col] = df[col].replace({'YES':'Yes','NO':'No','Y':'Yes','N':'No','yes':'Yes','no':'No'})

In [ ]:
# 7) Enforce domain-specific consistency: if PhoneService is No, MultipleLines must be No phone service

mask_no_phone = df['PhoneService'].eq('No')
df.loc[mask_no_phone, 'MultipleLines'] = 'No phone service'


In [ ]:
# 8) Convert key categoricals to category dtype (optional but helpful)

cat_cols = [
    'gender','Partner','Dependents','PhoneService','MultipleLines','InternetService',
    'OnlineSecurity','OnlineBackup','DeviceProtection','TechSupport','StreamingTV','StreamingMovies',
    'Contract','PaperlessBilling','PaymentMethod','Churn'
]
for col in cat_cols:
    df[col] = df[col].astype('category')


In [ ]:
# 9) Quick quality report
print(df.head())
print(df.shape)
print(missing_total_before)
print(missing_total_after)
print(rows_before)
print(rows_after)


In [ ]:
#Save cleaned/transformed dataset for download
out_path = 'Customer_Churn_cleaned.csv'
df.to_csv(out_path, index=False)
print(out_path)

# Quick summary stats for key numeric fields
num_cols = [c for c in ['tenure','monthly_charges','total_charges','avg_charges_per_month'] if c in df.columns]
print(df[num_cols].describe())

Customer_Churn_cleaned.csv
            tenure
count  7043.000000
mean     32.371149
std      24.559481
min       0.000000
25%       9.000000
50%      29.000000
75%      55.000000
max      72.000000


In [ ]:
# %pip install cryptography

from sqlalchemy import create_engine

# MySQL connection
username = "root"
password = "surya123"
host = "localhost"
port = "3306"
database = "customer_churn"

engine = create_engine(f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}")

# Write DataFrame to MySQL
table_name = "customer"   # choose any table name
df.to_sql(table_name, engine, if_exists="replace", index=False)

# Read back sample
pd.read_sql("SELECT * FROM customer LIMIT 5;", engine)

,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes
